# 6.5 总结：DeepFM、DIN 与 DIEN 排序

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

读取本章节每个独立算法 Notebook 的实际结果产物，在统一口径下比较和选型。

## Setup

本 Notebook 的默认真实数据是 **Amazon Reviews Electronics 5-core：DIN/DIEN 公开复现实验数据**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** DeepFM · DIN · DIEN 原始论文

In [ ]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "amazon-electronics"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 开篇回顾

本章节固定为 **开篇导读 → 独立算法教程 → 结果总结**。每篇算法 Notebook 都包含论文、数学、数据、训练、推理、测试与讨论；本页不重新训练，也不手填数字。

DeepFM 处理静态 field 交互；DIN 让历史随候选变化；DIEN 再利用次序。复杂模型不保证胜出，AUC、LogLoss、GAUC 与 P99 必须同时评估。

## 论文关联关系：后一篇在修补前一篇的哪块短板？

三篇论文沿同一条 CTR 排序线各自补一块短板：DeepFM 解决“静态特征交互要不要手工”，DIN 解决“用户表示与候选无关”，DIEN 解决“历史没有次序、行为不等于兴趣”。下表中的“交接问题”比发表年份更重要。

In [ ]:
import pandas as pd
paper_relationships = pd.DataFrame([{'paper': 'DeepFM (2017)', 'starts_from': 'Wide & Deep 的 wide 侧依赖人工交叉特征', 'user_representation': '静态 field embedding：一阶线性 + FM 二阶 + DNN 高阶三路相加', 'training_signal': '曝光样本的点击/未点击 BCE', 'serving_shape': '查 embedding 后 FM 与 DNN 并行打分', 'hands_to_next': '只有静态交互；用户表示不含行为历史、与候选无关'}, {'paper': 'DIN (2018)', 'starts_from': '固定长度用户向量表达不下多样兴趣', 'user_representation': '候选感知的历史加权和（一个候选一个向量）', 'training_signal': 'BCE + mini-batch aware 正则 + Dice 激活', 'serving_shape': '每个候选重算历史注意力再进 MLP', 'hands_to_next': '利用相关性但忽略行为先后次序；行为被直接当作兴趣'}, {'paper': 'DIEN (2019)', 'starts_from': '行为不等于兴趣，且兴趣随时间演化、会漂移', 'user_representation': 'GRU 兴趣状态序列 + 候选感知 AUGRU 演化末状态', 'training_signal': 'BCE + 逐步辅助损失（下一真实行为 vs 负采样）', 'serving_shape': '串行 GRU/AUGRU，需 kernel 融合与压缩降延迟', 'hands_to_next': '串行结构限制长序列；长期兴趣留存与多峰演化仍开放'}])
display(paper_relationships)

## Results

读取 results 目录。若缺文件，请先按章节顺序执行算法 Notebook。

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
result_dir=ARTIFACT_ROOT/'results'/'chapter_6'; files=sorted(result_dir.glob('*.json'))
assert len(files)==3,f'期望 3 个结果，实际 {[p.name for p in files]}'
records=[]
for path in files: records.extend(json.loads(path.read_text(encoding='utf-8'))['records'])
comparison=pd.DataFrame(records); display(comparison.round(4)); print('数据来源:',[p.name for p in files])

In [ ]:
from matplotlib import font_manager

# Matplotlib 默认的 DejaVu Sans 不包含中文字形。优先选择容器中安装的
# Noto CJK；在精简宿主机上找不到中文字体时，退回纯 ASCII 的 Notebook 编号，
# 从根源避免 missing glyph 警告，而不是用 warnings.filterwarnings 隐藏它。
cjk_candidates = ('Noto Sans CJK SC', 'Noto Sans CJK JP', 'Microsoft YaHei', 'SimHei', 'PingFang SC')
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
cjk_font = next((name for name in cjk_candidates if name in installed_fonts), None)
if cjk_font:
    plt.rcParams['font.sans-serif'] = [cjk_font, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
chart_labels = comparison.algorithm if cjk_font else comparison.source_notebook
print('图表字体:', cjk_font or 'ASCII fallback（宿主机未安装 CJK 字体）')

fig,ax=plt.subplots(figsize=(max(7,len(comparison)*1.5),3.8))
bars=ax.bar(chart_labels,comparison.primary_value,color='#7ca832')
ax.set(title='Primary metrics from executed notebooks',ylabel='metric value',ylim=(0,max(1.0,comparison.primary_value.max()*1.18)))
ax.tick_params(axis='x',rotation=12)
for bar,value in zip(bars,comparison.primary_value): ax.text(bar.get_x()+bar.get_width()/2,value,f'{value:.3f}',ha='center',va='bottom')
plt.tight_layout(); plt.show()

## 与原论文的可比性审计

下表逐项核对本教程实际产物与论文表格。**“不可直接比较”不是回避差距**：候选集合、样本规模、切分和指标任一不同，数值相减就没有统计含义。这里同时指出当前实验能证明什么、不能证明什么。

In [ ]:
paper_audit = pd.DataFrame([{'algorithm': 'DeepFM', 'source_notebook': '6_2_deepfm', 'paper_result': 'Criteo AUC=0.8007；LogLoss=0.45083', 'paper_protocol': 'Criteo 约 4,500 万条点击记录、13 连续 + 26 离散特征，随机 9:1；400-400-400 MLP、dropout 0.5、FM 维度 10；Company* 约 10 亿条记录、7 天训练 + 次日测试', 'verdict': '不可直接比较：数据集、特征口径与切分完全不同；教程数值来自 results JSON，只验证共享 embedding 的三路结构可训练。'}, {'algorithm': 'DIN', 'source_notebook': '6_3_din', 'paper_result': 'Amazon Electronics AUC=0.8818（Dice 版 0.8871）；Alibaba AUC=0.6083（MBA+Dice）', 'paper_protocol': 'Amazon 192,403 用户、1,689,188 样本；阿里约 20 亿训练样本；用户加权 AUC（GAUC）；线上 A/B CTR +10.0%、RPM +3.8%', 'verdict': '论文用 GAUC 且候选为广告；教程在 KuaiRand 曝光上测量，数值不可相减，只验证 target-aware 路径可运行。'}, {'algorithm': 'DIEN', 'source_notebook': '6_4_dien', 'paper_result': 'Electronics AUC=0.7792；Books AUC=0.8453；工业 AUC=0.6541', 'paper_protocol': '公开集 192,403/603,668 用户、各重复 5 次；工业 70 亿样本、历史截断 50；在线 A/B CTR +20.7%、eCPM +17.1%', 'verdict': '不可跨数据集比较绝对值；教程只验证 GRU+辅助损失+AUGRU 链路可运行。'}])
live = comparison.copy()
live['tutorial_result'] = live.apply(
    lambda row: f"{row.primary_metric}={row.primary_value:.4f}；{row.secondary_metric}={row.secondary_value:.4f}", axis=1
)
paper_audit = paper_audit.merge(live[['source_notebook', 'tutorial_result']], on='source_notebook', how='left')
display(paper_audit[['algorithm', 'tutorial_result', 'paper_result', 'paper_protocol', 'verdict']])
print('结论：教程数值来自本次 results JSON；smoke/迁移实验不是 paper reproduction。')

## 未来发展：沿着什么约束继续前进？

未来路线不是一味堆更深网络，而是逐项解除当前系统约束。表格从左到右给出本章已经走过的变化和仍待验证的方向；最后一列是研究/工程问题，不是预告一定会获得线上提升。

In [ ]:
future = pd.DataFrame([['交互对象', '静态 field 低阶/高阶交互', '历史行为 × 候选交互', '兴趣状态 × 候选逐步交互', '多模态内容、上下文与图的统一交互'], ['用户表示', '无显式用户向量（field 拼接）', '候选感知单向量', '候选感知演化状态', '长期兴趣记忆 + 多兴趣演化'], ['训练信号', '点击 BCE', 'BCE + MBA 正则 + Dice', 'BCE + 序列辅助损失', '去偏、多行为目标、自监督预训练'], ['服务成本', '一次查表 + 两路前向', '每候选重算注意力', '串行 GRU，108→32 维压缩', '长序列稀疏化、状态缓存、蒸馏'], ['必须补测', 'AUC + LogLoss + 校准', 'GAUC + 候选批量延迟', '在线 A/B + P99 延迟', '公平性、生态长期收益与成本 ROI']], columns=['dimension', 'DeepFM line', 'DIN line', 'DIEN line', 'next questions'])
display(future)

## Takeaways

DeepFM 处理静态 field 交互；DIN 让历史随候选变化；DIEN 再利用次序。复杂模型不保证胜出，AUC、LogLoss、GAUC 与 P99 必须同时评估。

先固定业务阶段和候选口径，再比较主指标、辅助指标、baseline 与系统成本。smoke 数值用于代码回归和学习，不能跨数据或跨公司宣称优劣。

## Checks

In [ ]:
assert len(comparison)==3
assert comparison.source_notebook.nunique()==3
assert comparison.primary_value.between(0,1).all()
print('PASS：总结完全来自独立 Notebook 的执行产物。')

## Next Steps

在相同完整数据、时间切分、负样本和候选集上重跑；加入效果—延迟—成本三维表，再决定是否进入线上 A/B。